# MAPPO Curriculum

This notebook trains the first curriculum stage for `AntByteForagingEnv`: ants learn to reach cookie sources, pick up bites, return to the hub, and write configurable tile values into the environment. The trainer predicts both movement and write-value actions for every ant; `WRITE_BITS = 1` is the current curriculum setting and can be raised to 3, 5, or up to 8 later. Training now uses the pure JAX MAPPO path: JAX environment rollouts, JIT-compiled GAE/PPO updates, local actor observations, and a centralized critic.

The training cell below grows the map progressively. It pads observations to the largest scheduled map, so the same actor/critic checkpoint can continue from smaller maps to larger maps. Each episode randomizes the colony location and uses multiple random cookie source locations.

Install the notebook/training extras from the repo root if needed:

```bash
python -m pip install -e ".[jax-cuda13,notebooks]"
```


In [ ]:
from pathlib import Path
import os
import sys

# These must be set before importing JAX in this kernel.
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
os.environ.setdefault("XLA_PYTHON_CLIENT_MEM_FRACTION", "0.65")
if "jax" in sys.modules:
    print("Restart the kernel before rerunning training; JAX was already imported.")

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)
SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

{
    "project_root": PROJECT_ROOT,
    "jax_preallocate": os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"],
    "jax_memory_fraction": os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"],
}


In [ ]:
import sys

try:
    import jax  # noqa: F401
    import jax.numpy as jnp  # noqa: F401
    import tqdm  # noqa: F401
except ModuleNotFoundError as exc:
    missing = exc.name or "jax/notebook extras"
    raise ModuleNotFoundError(
        f"Missing {missing!r} in this notebook kernel ({sys.executable}). "
        "Install the JAX notebook extras from the repo root with: "
        f'{sys.executable} -m pip install -e ".[jax-cuda13,notebooks]"'
    ) from exc


In [ ]:
from tqdm.auto import tqdm

from ant_byte_env.rendering import render_checkpoint
from ant_byte_env.training.jax_mappo import main
from ant_byte_env.vault import create_vault_entry


## Quick Smoke Run

Run this first to make sure the notebook kernel can import the repo, create the JAX environment/trainer path, and complete one tiny JAX MAPPO update.


In [ ]:
smoke_metrics = main(
    [
        "--total-timesteps", "8",
        "--num-envs", "1",
        "--num-steps", "4",
        "--num-minibatches", "1",
        "--update-epochs", "1",
        "--width", "4",
        "--height", "4",
        "--num-ants", "1",
        "--food-count", "1",
        "--max-steps", "8",
        "--write-bits", "1",
        "--hidden-size", "16",
        "--seed", "11",
        "--quiet",
    ]
)
smoke_metrics


## Progressive Map Curriculum

Each stage resumes from the previous stage checkpoint. Keep `--obs-width` and `--obs-height` equal to the largest scheduled map, otherwise the padded critic input size will not match the saved checkpoint.

The default schedule trains every square map size from `4x4` through `15x15`. It increases `cookie_distance`, uses a moderate food count, grows the number of food sources, randomizes the hub, and keeps `WRITE_BITS = 1` for now.

Each stage trains for exactly `GLOBAL_UPDATE_CAP` JAX MAPPO updates and saves a checkpoint. The first visible update for a fresh shape includes XLA compilation, so the progress bar moves only after that update returns. Rollout rendering is in a separate optional cell so video export cannot interfere with the JAX training loop.


In [ ]:
RUN_DIR = PROJECT_ROOT / "runs" / "notebooks" / "forage_curriculum"
CHECKPOINT_DIR = RUN_DIR / "checkpoints"
MEDIA_DIR = RUN_DIR / "media"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
MEDIA_DIR.mkdir(parents=True, exist_ok=True)


def curriculum_food_count(size):
    return 2 + max(0, size - 4)


def curriculum_food_sources(size):
    return min(curriculum_food_count(size), max(2, size // 2))


CURRICULUM_STAGES = [
    {
        "name": f"{size}x{size}",
        "width": size,
        "height": size,
        "food_count": curriculum_food_count(size),
        "food_sources": curriculum_food_sources(size),
        "cookie_distance": min(1 + (size - 4) // 2, size // 2),
        "max_steps": max(48, 4 * size * size),
    }
    for size in range(4, 16)
]

MAX_WIDTH = max(stage["width"] for stage in CURRICULUM_STAGES)
MAX_HEIGHT = max(stage["height"] for stage in CURRICULUM_STAGES)
NUM_ENVS = 16
NUM_STEPS = 80
UPDATE_TIMESTEPS = NUM_ENVS * NUM_STEPS
GLOBAL_UPDATE_CAP = 2000
ACTOR_VISION_RADIUS = 2
WRITE_BITS = 1
print(f"JAX device: {jax.devices()[0]}")
print(f"JAX preallocate: {os.environ.get('XLA_PYTHON_CLIENT_PREALLOCATE')}")
print(f"JAX memory fraction: {os.environ.get('XLA_PYTHON_CLIENT_MEM_FRACTION')}")

COMMON_ARGS = [
    "--num-envs", str(NUM_ENVS),
    "--num-steps", str(NUM_STEPS),
    "--num-minibatches", "4",
    "--update-epochs", "4",
    "--obs-width", str(MAX_WIDTH),
    "--obs-height", str(MAX_HEIGHT),
    "--actor-vision-radius", str(ACTOR_VISION_RADIUS),
    "--write-bits", str(WRITE_BITS),
    "--num-ants", "1",
    "--random-food",
    "--random-hub",
    "--pickup-bonus", "0.25",
    "--distance-bonus", "0.02",
    "--hidden-size", "128",
    "--seed", "1",
    "--quiet",
]


## Train Curriculum Checkpoints

This cell only trains and saves checkpoints. Render and archive rollouts afterwards with the optional cell below.


In [ ]:
stage_metrics = []
stage_checkpoint_paths = []
previous_checkpoint = None

for stage_index, stage in enumerate(CURRICULUM_STAGES, start=1):
    print(f"Training stage {stage_index}/{len(CURRICULUM_STAGES)}: {stage['name']}")
    print("First update for this shape may compile; progress starts after it returns.")
    checkpoint_path = CHECKPOINT_DIR / f"jax_mappo_forage_stage1_{stage['name']}.pkl"
    update_iterator = tqdm(
        range(1, GLOBAL_UPDATE_CAP + 1),
        total=GLOBAL_UPDATE_CAP,
        desc=f"{stage['name']}",
        bar_format="{desc}: {n_fmt}/{total_fmt} updates |{bar}| {elapsed}<{remaining} {postfix}",
        leave=True,
    )

    def record_progress(update_index, total_updates, train_metrics):
        del total_updates
        update_iterator.update(1)
        update_iterator.set_postfix(
            loss=f"{train_metrics['loss']:.3f}",
            ret=f"{train_metrics['episode_return']:.3f}",
        )
        stage_metrics.append(
            {
                **stage,
                **train_metrics,
                "stage_update": update_index,
                "global_update_cap": GLOBAL_UPDATE_CAP,
                "checkpoint": str(checkpoint_path),
            }
        )

    train_args = [
        *COMMON_ARGS,
        "--total-timesteps", str(UPDATE_TIMESTEPS * GLOBAL_UPDATE_CAP),
        "--width", str(stage["width"]),
        "--height", str(stage["height"]),
        "--food-count", str(stage["food_count"]),
        "--food-sources", str(stage["food_sources"]),
        "--cookie-distance", str(stage["cookie_distance"]),
        "--max-steps", str(stage["max_steps"]),
        "--save-model", str(checkpoint_path),
    ]
    if previous_checkpoint is not None:
        train_args.extend(["--load-model", str(previous_checkpoint)])

    try:
        final_train_metrics = main(train_args, progress_callback=record_progress)
    finally:
        update_iterator.close()

    stage_checkpoint_paths.append(checkpoint_path)
    print(f"Saved checkpoint to {checkpoint_path}")

    previous_checkpoint = checkpoint_path

FINAL_CHECKPOINT_PATH = previous_checkpoint
{
    "stage_metrics": stage_metrics,
    "stage_checkpoint_paths": stage_checkpoint_paths,
    "final_checkpoint_path": FINAL_CHECKPOINT_PATH,
    "final_train_metrics": final_train_metrics,
}


In [ ]:
def render_policy_rollout(checkpoint_path):
    checkpoint_path = Path(checkpoint_path)
    rollout_path = MEDIA_DIR / f"{checkpoint_path.stem}_rollout.gif"
    return render_checkpoint(checkpoint_path, rollout_path, backend="jax")


In [ ]:
policy_checkpoint_paths = [
    CHECKPOINT_DIR / f"jax_mappo_forage_stage1_{stage['name']}.pkl"
    for stage in CURRICULUM_STAGES
]
missing_checkpoints = [path for path in policy_checkpoint_paths if not path.exists()]
if missing_checkpoints:
    missing = "\n".join(str(path) for path in missing_checkpoints)
    raise FileNotFoundError(f"Train the missing stage policies before rendering:\n{missing}")

rollout_paths = [
    render_policy_rollout(path)
    for path in tqdm(policy_checkpoint_paths, desc="rendering policies")
]
vault_entry_path = create_vault_entry(
    vault_dir=RUN_DIR / "vault",
    title="JAX MAPPO curriculum policy rollouts",
    description="Rollout GIFs for each saved JAX MAPPO curriculum stage policy.",
    assets=rollout_paths,
    metadata={
        "stages": [stage["name"] for stage in CURRICULUM_STAGES],
        "checkpoint_paths": [str(path) for path in policy_checkpoint_paths],
        "rollout_paths": [str(path) for path in rollout_paths],
        "actor_vision_radius": ACTOR_VISION_RADIUS,
        "write_bits": WRITE_BITS,
        "global_update_cap": GLOBAL_UPDATE_CAP,
    },
)
{
    "rollout_paths": rollout_paths,
    "vault_entry_path": vault_entry_path,
}
